# Binary Search


## Topic overview

Halve the search space each step over a monotonic predicate.

## Pattern-recognition rules

- Search a sorted array.
- Search on the *answer* (parametric search).
- First/last occurrence via boundary shifting.

## Common data structures

- Sorted arrays
- Any monotonic predicate space

## Standard complexity expectations

- O(log n) per query.

## Common mistakes

- Off-by-one on `lo`/`hi` when converging.
- Overflow in `(lo+hi)//2` in non-Python languages.

## Original illustrative example

In [ ]:
# Replace with an ORIGINAL example. Do not paste external
# problem statements. See src/algorithms/ for reusable helpers.
example_input = []
example_expected = None

## Add solved problems below

Each new sub-section should follow the template in `../templates/notebook_template.ipynb`.

1. [Search a 2D Matrix](#search-a-2d-matrix)
2. [Reverse Nodes in k-Group](#reverse-nodes-in-k-group)


# Search a 2D Matrix

## Metadata

- Source: NeetCode / LeetCode 74
- Problem URL: https://leetcode.com/problems/search-a-2d-matrix/
- Difficulty: Medium
- Topic: Binary Search
- Date started: 2026-09-01
- Date solved: 2026-09-01
- Current mastery level: 1
- Last reviewed: 2026-09-01
- Next review:

## Problem statement in my own words

You get an `m x n` integer grid and a target number. Decide whether that
target appears anywhere in the grid.

Two sorting facts make this a binary-search problem, not a scan:

1. Every row is sorted left to right (non-decreasing).
2. The first value of each row is **greater than** the last value of the
   previous row.

Those two facts mean the whole matrix is globally sorted if you read it
row by row. Mentally flatten

```
[
    [ 1,  2,  4,  8],
    [10, 11, 12, 13],
    [14, 20, 30, 40]
]
```

and you get the ordinary sorted array

```
[1, 2, 4, 8, 10, 11, 12, 13, 14, 20, 30, 40]
```

Return `True` if `target` is present, otherwise `False`. The intended
bound is $O(\log(mn))$ — one binary search over the virtual 1-D array,
**without** allocating a flattened copy.

## Inputs, outputs, and constraints

- Inputs: `matrix` (`m` rows, `n` columns of ints) and `target` (int)
- Outputs: `bool` — whether `target` exists in `matrix`
- Constraints:
  - $1 \le m, n \le 100$
  - $-10^4 \le$ `matrix[i][j]`, `target` $\le 10^4$
  - each row is sorted non-decreasing
  - `matrix[r][0] > matrix[r-1][n-1]` for every row `r >= 1`

## Examples

| Input | Expected | Notes |
|---|---|---|
| `matrix = [[1,2,4,8],[10,11,12,13],[14,20,30,40]]`, `target = 10` | `True` | target sits at `[1][0]` |
| `matrix = [[1,2,4,8],[10,11,12,13],[14,20,30,40]]`, `target = 15` | `False` | 15 falls in the gap between 14 and 20 |

Same $3 \times 4$ grid in both examples. Flattened view:

```
index:  0  1  2  3   4   5   6   7   8   9  10  11
value: [1, 2, 4, 8, 10, 11, 12, 13, 14, 20, 30, 40]
```

## Initial observations

- A nested scan is correct but $O(mn)$ — too slow for the asked bound.
- Binary-searching **each row** is $O(m \log n)$. Better, still not
  $O(\log(mn))$.
- Because the matrix is globally sorted, it behaves like one sorted
  array of length $N = mn$. Binary search on that virtual array is
  $O(\log(mn))$.
- The only extra work is mapping a 1-D index `mid` back to `(row, col)`
  so we never allocate the flattened copy.

## Brute-force approach

### Why it works

Walk every cell. If any equals `target`, return `True`. The matrix
guarantees are unused; this is just exhaustive search.

### Implementation

In [ ]:
from typing import List


def brute_force(matrix: List[List[int]], target: int) -> bool:
    for row in matrix:
        for value in row:
            if value == target:
                return True
    return False

### Complexity

- Time: $O(mn)$
- Space: $O(1)$

## Optimized insight

Treat the matrix as one long sorted array of length $mn$, then run the
same binary-search loop you use on a 1-D sorted list.

**Do not actually flatten.** Building `flat` is already $O(mn)$ and
destroys the required $O(\log(mn))$ bound. Convert the virtual index
instead:

$$
\text{row} = \left\lfloor \frac{\text{mid}}{n} \right\rfloor, \qquad
\text{col} = \text{mid} \bmod n
$$

In Python, with `n = cols`:

```python
row = mid // cols
col = mid % cols
```

Division says which row you have reached. Remainder says how far into
that row you have moved.

For $n = 4$:

```
Virtual index:

  0   1   2   3
  4   5   6   7
  8   9  10  11

Matrix value:

  1   2   4   8
 10  11  12  13
 14  20  30  40
```

`mid = 6` → `row = 6 // 4 = 1`, `col = 6 % 4 = 2` → `matrix[1][2] = 12`.

`mid = 10` → `row = 10 // 4 = 2`, `col = 10 % 4 = 2` → `matrix[2][2] = 30`.

## Optimized approach

1. Set `left = 0`, `right = m * n - 1`.
2. While `left <= right`:
   - `mid = (left + right) // 2`
   - read `value = matrix[mid // n][mid % n]`
   - equal → `True`
   - too small → drop the left half (`left = mid + 1`)
   - too large → drop the right half (`right = mid - 1`)
3. If the window empties, return `False`.

### Step-by-step trace

Target `10` on the $3 \times 4$ example. Search space starts as
`[0, 11]`.

| Step | left | right | mid | (row, col) | value | Note |
|---|---|---|---|---|---|---|
| 1 | 0 | 11 | 5 | (1, 1) | 11 | 11 > 10 → `right = 4` |
| 2 | 0 | 4 | 2 | (0, 2) | 4 | 4 < 10 → `left = 3` |
| 3 | 3 | 4 | 3 | (0, 3) | 8 | 8 < 10 → `left = 4` |
| 4 | 4 | 4 | 4 | (1, 0) | 10 | found → `True` |

Flattened view of the same process:

```
[1, 2, 4, 8, 10, 11, 12, 13, 14, 20, 30, 40]
                ↑ mid=5 (11)  → drop right
[1, 2, 4, 8, 10]
       ↑ mid=2 (4)            → drop left
[8, 10]
 ↑ mid=3 (8)                  → drop left
[10]
  ↑ found
```

For `target = 15`, binary search closes the window on the gap
`14 < 15 < 20`. Then `left > right` and the answer is `False`.

In [ ]:
from typing import List


class Solution:
    def searchMatrix(self, matrix: List[List[int]], target: int) -> bool:
        rows = len(matrix)
        cols = len(matrix[0])

        left = 0
        right = rows * cols - 1

        while left <= right:
            mid = (left + right) // 2

            row = mid // cols
            col = mid % cols
            value = matrix[row][col]

            if value == target:
                return True
            if value < target:
                left = mid + 1
            else:
                right = mid - 1

        return False


def optimized(matrix: List[List[int]], target: int) -> bool:
    return Solution().searchMatrix(matrix, target)

## Complexity analysis

Let $N = mn$ be the number of cells.

Each comparison halves the remaining window:

$$
N \rightarrow N/2 \rightarrow N/4 \rightarrow \cdots \rightarrow 1
$$

so the number of iterations is $\log_2 N = \log(mn)$.

- Time: $O(\log(mn))$
- Space: $O(1)$ — only `left`, `right`, `mid`, `row`, `col`, `value`

## Edge cases

- Single cell: `[[7]]` — window is `[0, 0]`; one comparison decides it
- Target smaller than `matrix[0][0]` — search walks left and empties
- Target larger than `matrix[-1][-1]` — search walks right and empties
- Target equals a row boundary (`8` or `10` in the example) — the
  `mid // cols` / `mid % cols` map still lands on the correct cell
- Value in the gap between two rows (`15` between 14 and 20) — `False`
- Negative numbers are legal under the constraints; the same loop works
  because order, not sign, is what binary search needs

## Testing

In [ ]:
matrix = [
    [1, 2, 4, 8],
    [10, 11, 12, 13],
    [14, 20, 30, 40],
]

assert optimized(matrix, 10) is True
assert optimized(matrix, 15) is False
assert optimized(matrix, 1) is True
assert optimized(matrix, 40) is True
assert optimized(matrix, 8) is True
assert optimized(matrix, 14) is True
assert optimized(matrix, 0) is False
assert optimized(matrix, 41) is False
assert optimized([[7]], 7) is True
assert optimized([[7]], 3) is False
assert brute_force(matrix, 10) is True
assert brute_force(matrix, 15) is False

print("Search a 2D Matrix checks passed")

## Alternative approaches

- Scan every cell: $O(mn)$ time, $O(1)$ space. Correct, too slow.
- Binary search each row: $O(m \log n)$. Uses the per-row sort but
  ignores the global order between rows.
- Binary search the first column to pick a row, then binary search that
  row: $O(\log m + \log n) = O(\log(mn))$. Same bound, two searches.
  The virtual-index version is one loop and the same idea.
- Materialize `flat` then binary search it: still $O(\log(mn))$
  comparisons, but $O(mn)$ time and space to build `flat`. Do not do this.

## Mistakes I made

- Forgetting that actually flattening the matrix is already linear.
- Using `mid % rows` instead of `mid % cols` when mapping the index.
- Exclusive right bound (`right = mn`) with a `left <= right` loop —
  pick one convention and stick to it. Here: inclusive `[left, right]`.

## Pattern recognition

This is 1-D binary search plus an index encoding. Whenever a 2-D
structure is **row-major and globally sorted**, a flat index

```text
index = row * cols + col
```

and its inverse

```text
row = index // cols
col = index % cols
```

let you reuse the 1-D algorithm.

## Related problems

- Binary Search (1-D) — same loop, no `(row, col)` map
- Search a 2D Matrix II — rows and columns sorted, but **not** globally
  sorted across row boundaries; the flatten trick does not apply
- Find First and Last Position of Element in Sorted Array — same search
  space idea, different boundary policy
- Koko Eating Bananas / capacity problems — binary search on the
  *answer*, not on array indices

## Real-world or engineering connection

Row-major layouts (images, matrices, C arrays) already store a 2-D grid
as one contiguous buffer. The `index // cols`, `index % cols` map is
exactly how those buffers are addressed. Here we use that addressing so
we can binary-search the buffer without copying it.

## Final takeaways

The two lines that carry the whole problem:

```python
row = mid // cols
col = mid % cols
```

Same binary-search skeleton as the 1-D problem. The matrix is only a
view over a sorted array of length $mn$.

## Reattempt log

| Date | Mastery | Notes |
|---|---|---|
| 2026-09-01 | 1 | First write-up: virtual flatten + binary search |

# Reverse Nodes in k-Group

## Metadata

- Source: NeetCode / LeetCode 25
- Problem URL: https://leetcode.com/problems/reverse-nodes-in-k-group/
- Difficulty: Hard
- Topic: Linked Lists (pointer reversal)
- Date started: 2026-09-01
- Date solved: 2026-09-01
- Current mastery level: 1
- Last reviewed: 2026-09-01
- Next review:

When $k = 2$, this is Swap Nodes in Pairs. The extra work is grouping,
stopping early when a leftover suffix is shorter than $k$, and
reconnecting each reversed block without losing pointers.

## Problem statement in my own words

You are given the head of a singly linked list and a positive integer
$k$. Reverse the list **in groups of exactly $k$ nodes**, then return
the new head.

Rules:

- Only complete groups of $k$ are reversed.
- If the leftover suffix has fewer than $k$ nodes, leave that suffix
  in its original order.
- Rewire the nodes. Do not swap values inside nodes.

Example with $k = 2$:

```
1 → 2 → 3 → 4 → 5
[1 → 2] [3 → 4] [5]
[2 → 1] [4 → 3] [5]
2 → 1 → 4 → 3 → 5
```

Example with $k = 3$:

```
1 → 2 → 3 → 4 → 5
[1 → 2 → 3] [4 → 5]
[3 → 2 → 1] [4 → 5]
3 → 2 → 1 → 4 → 5
```

## Inputs, outputs, and constraints

- Inputs: `head` of a singly linked list, and `k`
- Outputs: head of the modified list
- Constraints:
  - $n$ = number of nodes
  - $1 \le k \le n \le 5000$
  - $0 \le$ `Node.val` $\le 1000$
  - $k$ is always at most the length of the list

## Examples

| Input | Expected | Notes |
|---|---|---|
| `head = [1,2,3,4,5]`, `k = 2` | `[2,1,4,3,5]` | two complete pairs; leftover `5` stays |
| `head = [1,2,3,4,5]`, `k = 3` | `[3,2,1,4,5]` | one complete triple; leftover `4 → 5` stays |

## Initial observations

- Reversing a linked list is the easy part. The hard part is reversing
  **exactly $k$ nodes**, hooking that block back to the previous group
  and the unused suffix, then repeating.
- The first group's new head becomes the answer head. A dummy node
  sitting before `head` makes the first group look like every later
  group, so you do not special-case it.
- Walking $k$ steps from the node before a group tells you whether a
  full group exists. If that walk falls off the list, stop.
- The one trick in the reversal loop: initialize `prev` to the node
  *after* the group, not `None`. The new tail then already points at
  the remainder.

## Brute-force approach

### Why it works

Collect every node into an array, reverse each complete chunk of $k$
nodes, then rewire `next` pointers from the new order. Correct, but it
uses $O(n)$ extra space.

### Implementation

In [ ]:
from typing import List, Optional


class ListNode:
    def __init__(self, val: int = 0, next: Optional["ListNode"] = None):
        self.val = val
        self.next = next


def to_list(head: Optional[ListNode]) -> List[int]:
    values: List[int] = []
    curr = head
    while curr is not None:
        values.append(curr.val)
        curr = curr.next
    return values


def from_list(values: List[int]) -> Optional[ListNode]:
    dummy = ListNode(0)
    curr = dummy
    for value in values:
        curr.next = ListNode(value)
        curr = curr.next
    return dummy.next


def brute_force_k_group(head: Optional[ListNode], k: int) -> Optional[ListNode]:
    nodes: List[ListNode] = []
    curr = head
    while curr is not None:
        nodes.append(curr)
        curr = curr.next

    for start in range(0, len(nodes) - len(nodes) % k, k):
        left, right = start, start + k - 1
        while left < right:
            nodes[left], nodes[right] = nodes[right], nodes[left]
            left += 1
            right -= 1

    for i in range(len(nodes) - 1):
        nodes[i].next = nodes[i + 1]
    if nodes:
        nodes[-1].next = None
    return nodes[0] if nodes else None

### Complexity

- Time: $O(n)$
- Space: $O(n)$ for the node array

## Optimized insight

Use in-place reversal plus four named pointers. For a segment

```
... → A → 1 → 2 → 3 → B → ...
```

with $k = 3$:

```
      groupPrev
          ↓
... → A → 1 → 2 → 3 → B → ...
                    ↑    ↑
                   kth  groupNext
```

After reversing that group:

```
... → A → 3 → 2 → 1 → B → ...
          ↑         ↑
        new        new
       start       tail
```

`kth` becomes the new head of the group. The old first node becomes
the new tail, and therefore the `groupPrev` of the *next* group.

## Why a dummy node

Reversing the first group changes the real head (`1 → 2` becomes
`2 → 1`). If dummy sits in front:

```
dummy → 1 → 2 → 3 → 4
```

then every group has a predecessor, including the first. Return
`dummy.next` at the end.

## Optimized approach

1. `dummy.next = head`, `groupPrev = dummy`.
2. Repeat:
   - Walk $k$ steps from `groupPrev` to find `kth`.
   - If `kth` is missing, leftover suffix is shorter than $k$ — stop.
   - `groupNext = kth.next`.
   - Reverse the open interval from `groupPrev.next` up to (not
     including) `groupNext`, starting with `prev = groupNext`.
   - `oldStart = groupPrev.next` (this is the new tail).
   - `groupPrev.next = kth` (this is the new head of the group).
   - `groupPrev = oldStart`.
3. Return `dummy.next`.

### The four pointers

| Pointer | Job |
|---|---|
| `groupPrev` | node immediately before the current group |
| `kth` | last node of the current group |
| `groupNext` | first node after the current group |
| `curr` | node currently being reversed |

### Step-by-step trace — $k = 2$

Start:

```
dummy → 1 → 2 → 3 → 4 → 5
  ↑
groupPrev
```

First group: `kth = 2`, `groupNext = 3`. Reverse with `prev = 3`,
`curr = 1`.

| Step | temp | rewired | list after the write |
|---|---|---|---|
| 1 | 2 | `1.next = 3` | `1 → 3 → 4 → 5`, `curr` moves to 2 |
| 2 | 3 | `2.next = 1` | `2 → 1 → 3 → 4 → 5`, `curr` hits `groupNext` |

Reconnect dummy to `kth`:

```
dummy → 2 → 1 → 3 → 4 → 5
              ↑
          groupPrev (old start)
```

Second group: `kth = 4`, `groupNext = 5`. Reverse `3 → 4` into
`4 → 3`:

```
dummy → 2 → 1 → 4 → 3 → 5
```

Third group is only `5`. `getKth` returns `None`. Stop.

Result: $2 \rightarrow 1 \rightarrow 4 \rightarrow 3 \rightarrow 5$.

### Trace — $k = 3$

Complete group `[1 → 2 → 3]` reverses to `[3 → 2 → 1]`. Suffix
`4 → 5` has length $2 < k$, so it stays.

Result: $3 \rightarrow 2 \rightarrow 1 \rightarrow 4 \rightarrow 5$.

In [ ]:
class Solution:
    def reverseKGroup(self, head: Optional[ListNode], k: int) -> Optional[ListNode]:
        dummy = ListNode(0, head)
        groupPrev = dummy

        while True:
            kth = self.getKth(groupPrev, k)
            if kth is None:
                break

            groupNext = kth.next

            # prev = groupNext (not None) so the new tail already
            # points at the unused suffix.
            prev = groupNext
            curr = groupPrev.next

            while curr is not groupNext:
                temp = curr.next
                curr.next = prev
                prev = curr
                curr = temp

            oldGroupStart = groupPrev.next
            groupPrev.next = kth
            groupPrev = oldGroupStart

        return dummy.next

    def getKth(self, curr: Optional[ListNode], k: int) -> Optional[ListNode]:
        while curr is not None and k > 0:
            curr = curr.next
            k -= 1
        return curr


def optimized_k_group(head: Optional[ListNode], k: int) -> Optional[ListNode]:
    return Solution().reverseKGroup(head, k)

## Complexity analysis

Each node is visited a constant number of times: once while locating
its group, once while reversing it (if it belongs to a complete group).

- Time: $O(n)$
- Space: $O(1)$ extra — `dummy`, `groupPrev`, `kth`, `groupNext`,
  `prev`, `curr`, `temp`. No auxiliary list of size $n$.

That is the strongest form of the problem (in-place, constant extra
memory).

## Edge cases

- $k = 1$: every group is a single node; the list is unchanged
- $k = n$: the whole list reverses
- leftover suffix shorter than $k$: must stay in original order
- two nodes, $k = 2$: reduces to a single swap
- values are irrelevant; only `next` pointers move

## Testing

In [ ]:
def run_k_group(values: List[int], k: int) -> List[int]:
    return to_list(optimized_k_group(from_list(values), k))


assert run_k_group([1, 2, 3, 4, 5], 2) == [2, 1, 4, 3, 5]
assert run_k_group([1, 2, 3, 4, 5], 3) == [3, 2, 1, 4, 5]
assert run_k_group([1, 2, 3], 1) == [1, 2, 3]
assert run_k_group([1, 2, 3, 4], 4) == [4, 3, 2, 1]
assert run_k_group([1], 1) == [1]
assert to_list(brute_force_k_group(from_list([1, 2, 3, 4, 5]), 2)) == [2, 1, 4, 3, 5]
assert to_list(brute_force_k_group(from_list([1, 2, 3, 4, 5]), 3)) == [3, 2, 1, 4, 5]

print("Reverse Nodes in k-Group checks passed")

## Alternative approaches

- Array of nodes, reverse chunks, rewire: $O(n)$ time, $O(n)$ space.
  Useful as a check, not the intended solution.
- Recursion: reverse the first $k$ nodes, then recurse on the rest.
  Same $O(n)$ time, but $O(n / k)$ stack space.
- Swap values in each group: forbidden by the problem statement.

## Mistakes I made

- Reversing the leftover suffix when $n$ is not a multiple of $k$.
- Starting the reversal with `prev = None`, which detaches the group
  from the rest of the list.
- Losing `groupPrev.next` before reconnecting — save `oldGroupStart`
  first.
- Off-by-one in `getKth`: it must move exactly $k$ times from the node
  *before* the group.

## Pattern recognition

Standard linked-list reversal:

```text
save next → reverse pointer → advance prev → advance curr
```

with one change:

```python
prev = groupNext   # not None
```

That single assignment is what stitches each reversed block back onto
the unused suffix.

## Related problems

- Reverse Linked List — the inner loop with `prev = None`
- Swap Nodes in Pairs — this problem with $k = 2$
- Reverse Linked List II — reverse one closed index range
- Rotate List — regroup pointers, no reversal of values' order in
  the same way

## Real-world or engineering connection

Chunked in-place reversal shows up any time a singly linked buffer
must be rewritten in fixed-size blocks (protocol frames, disk-page
lists) without allocating a second copy of the chain.

## Final takeaways

Do not memorize the whole function. Keep this loop:

```python
prev = groupNext
curr = groupPrev.next

while curr is not groupNext:
    temp = curr.next
    curr.next = prev
    prev = curr
    curr = temp
```

`kth` becomes the new head of the group. The original first node
becomes the new tail and the predecessor of the next group.

## Reattempt log

| Date | Mastery | Notes |
|---|---|---|
| 2026-09-01 | 1 | First write-up: dummy + getKth + reverse onto groupNext |